# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ответы, полученные рассуждением до проверочного запуска:

1. `1 1.3333333333333333`: один пропуск; после заполнения нулём среднее равно `(1 + 0 + 3) / 3`.
2. `2`: после удаления крайних пробелов и приведения к нижнему регистру получаем `a, a, b`.
3. `3`: `drop_duplicates()` вернул новую таблицу, но результат не присвоили обратно в `df`.
4. `1005`: убираем пробел внутри `1 000`, переводим строки в числа и складываем.
5. `[9, 10]`: `.dt.month` извлекает месяц каждой даты.


In [ ]:
import pandas as pd
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
toy = pd.DataFrame({"k": [1, 1, 2]})
toy.drop_duplicates()
print(len(toy))
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "", regex=False)).sum())
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())


1 1.3333333333333333
2
3
1005
[9, 10]


In [ ]:
from pathlib import Path
import pandas as pd

# Поддерживаются имя из LMS и имя после повторного скачивания.
candidates = [Path("orders.csv"), Path("orders (4).csv"),
              Path("upload/orders (4).csv"), Path("../orders.csv")]
PATH = next((p for p in candidates if p.exists()), None)
if PATH is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Положите orders.csv рядом с ноутбуком и запустите ячейки сначала.")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("Загрузите один CSV-файл orders.csv.")
    PATH = Path(csv_names[0])
print("Файл данных:", PATH.name)


Файл данных: orders.csv


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [ ]:
CITY = {"алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    data = pd.read_csv(path)
    print(f"Чтение: {len(data):,} строк")
    data["amount"] = pd.to_numeric(data["amount"].str.replace(" ", "", regex=False))
    assert data["amount"].notna().all(), "Есть непрочитанные суммы"
    print(f"Суммы в числа: {len(data):,} строк; пропусков amount: 0")
    data["city"] = data["city"].str.strip().str.lower().map(CITY)
    assert data["city"].notna().all(), "Добавьте неизвестные города в словарь"
    print(f"Города по словарю: {len(data):,} строк; городов: {data['city'].nunique()}")
    data = data.drop_duplicates().copy()
    assert data["order_id"].is_unique, "Остались разные строки с одинаковым order_id"
    print(f"Удаление дублей: {len(data):,} строк")
    data["order_date"] = pd.to_datetime(data["order_date"])
    assert data["order_date"].notna().all()
    data["month"] = data["order_date"].dt.month
    print(f"Разбор дат: {len(data):,} строк")
    data = data.loc[data["customer_id"] != "TEST"].copy()
    print(f"Исключение TEST: {len(data):,} строк")
    assert data["amount"].notna().all()
    assert data["city"].notna().all()
    assert data["order_id"].is_unique
    return data

raw = pd.read_csv(PATH)
df = clean(PATH)
print(f"Для менеджера: осталось {len(df):,} реальных заказов; отмены сохранены — "
      f"{(df['status'] == 'отменён').sum():,} заказов.")


Чтение: 5,463 строк
Суммы в числа: 5,463 строк; пропусков amount: 0
Города по словарю: 5,463 строк; городов: 6
Удаление дублей: 5,283 строк
Разбор дат: 5,283 строк
Исключение TEST: 5,280 строк
Для менеджера: осталось 5,280 реальных заказов; отмены сохранены — 541 заказов.


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [ ]:
def revenue(data):
    paid = data["status"] == "оплачен"
    month = pd.to_datetime(data["order_date"]).dt.month
    sep = data.loc[paid & (month == 9), "amount"].sum()
    octo = data.loc[paid & (month == 10), "amount"].sum()
    return sep, octo, (octo / sep - 1) * 100

rows = []
for city in ["Все города"] + sorted(df["city"].unique().tolist()):
    part = df if city == "Все города" else df.loc[df["city"] == city]
    sep, octo, change = revenue(part)
    rows.append({"Город": city, "Сентябрь, ₸": sep, "Октябрь, ₸": octo, "Изменение, %": change})
report = pd.DataFrame(rows)
print(report.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
worst = report.iloc[1:].sort_values("Изменение, %").iloc[0]
sep, octo, change = revenue(df)
print(f"Для менеджера: оплаченная выручка {sep:,.0f} → {octo:,.0f} ₸, {change:.2f}%. "
      f"Сильнее всего падает {worst['Город']}: {worst['Изменение, %']:.2f}%.")
print("Итог совпадает с контрольным значением урока 4: −2,3% при округлении. "
      "Сам файл урока 4 не приложен, поэтому его таблицу напрямую не сравниваем.")


     Город   Сентябрь, ₸    Октябрь, ₸  Изменение, %
Все города 35,243,410.00 34,419,520.00         -2.34
    Актобе  2,576,960.00  2,540,420.00         -1.42
    Алматы 14,501,070.00 14,643,020.00          0.98
    Астана  9,034,220.00  8,850,530.00         -2.03
    Атырау    923,990.00    969,860.00          4.96
 Караганда  3,648,520.00  3,371,680.00         -7.59
   Шымкент  4,558,650.00  4,044,010.00        -11.29
Для менеджера: оплаченная выручка 35,243,410 → 34,419,520 ₸, -2.34%. Сильнее всего падает Шымкент: -11.29%.
Итог совпадает с контрольным значением урока 4: −2,3% при округлении. Сам файл урока 4 не приложен, поэтому его таблицу напрямую не сравниваем.


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# Промежуточный числовой расчёт нужен, чтобы проверить журнал ниже.
stage = raw.copy()
stage["amount"] = pd.to_numeric(stage["amount"].str.replace(" ", "", regex=False))
for label, data in [("Исходные данные", stage),
                    ("Без дублей", stage.drop_duplicates()),
                    ("Без дублей и TEST", df)]:
    s, o, delta = revenue(data)
    print(f"{label}: {len(data):,} строк; {s:,.0f} → {o:,.0f} ₸; {delta:.2f}%")
no_test = stage.loc[stage["customer_id"] != "TEST"]
print(f"Альтернативный порядок: сначала TEST → {revenue(no_test)[2]:.2f}%, "
      f"затем дубли → {revenue(no_test.drop_duplicates())[2]:.2f}%")


Исходные данные: 5,463 строк; 57,489,850 → 44,419,510 ₸; -22.74%
Без дублей: 5,283 строк; 55,243,390 → 44,419,510 ₸; -19.59%
Без дублей и TEST: 5,280 строк; 35,243,410 → 34,419,520 ₸; -2.34%
Альтернативный порядок: сначала TEST → -8.19%, затем дубли → -2.34%


| Шаг очистки | Убрано строк | Осталось строк | Выручка октября к сентябрю |
|---|---:|---:|---:|
| Исходный расчёт менеджера, суммы уже переведены в числа | 0 | 5 463 | −22,74% |
| Суммы: убрать пробелы, преобразовать тип | 0 | 5 463 | −22,74% |
| Города: strip → lower → словарь | 0 | 5 463 | −22,74% |
| Удалить полные дубли | 180 | 5 283 | −19,59% |
| Преобразовать даты | 0 | 5 283 | −19,59% |
| Исключить TEST | 3 | 5 280 | −2,34% |

**Менеджеру:** сильнее всего картину исказили тестовые заказы: при таком порядке очистки их исключение меняет результат на 17,26 процентного пункта; отменённые заказы сохраняем, а выручку считаем только по оплаченным.

Если сначала убрать TEST, получится **−22,74% → −8,19% → −2,34%** после дублей. Промежуточный эффект зависит от порядка, итог одинаков. Удаление дублей ничего не говорит об отменах: это отдельные признаки.


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
missing = df["delivery_days"].isna()
print("Всего пропусков delivery_days:", missing.sum())
print("Пропуски по городам:\n", df.loc[missing, "city"].value_counts().to_string())
print("Пропуски по статусам:\n", df.loc[missing, "status"].value_counts().to_string())
# Число пропусков зависит и от размера города, поэтому покажем также долю.
for city in sorted(df["city"].unique()):
    part = df.loc[df["city"] == city, "delivery_days"]
    print(f"{city}: {part.isna().sum()} из {len(part)}, {part.isna().mean():.2%}")
for status in sorted(df["status"].unique()):
    part = df.loc[df["status"] == status, "delivery_days"]
    print(f"{status}: {part.isna().sum()} из {len(part)}, {part.isna().mean():.2%}")
d = df.loc[df["city"] == "Шымкент", "delivery_days"]
print(f"Шымкент: всего {len(d)}, известных сроков {d.notna().sum()}, "
      f"пропусков {d.isna().sum()}, медиана {d.median():.0f} дней.")
for method, values in [("dropna", d.dropna()),
                       ("fillna(медиана)", d.fillna(d.median())),
                       ("fillna(0)", d.fillna(0))]:
    print(f"{method}: {(values <= 2).sum()} / {len(values)} = {(values <= 2).mean():.2%}")
print("Для менеджера: честная доля среди заказов с известным сроком — 29,84% (225/754). "
      "По 22 заказам срок неизвестен; это не оценка безусловной доли всех заказов. "
      "Медиана 5 делает пропуски медленными (28,99%), а ноль — быстрыми (31,83%). "
      "Ни то ни другое не подтверждено данными; для SLA нужны фактические сроки завершённых доставок.")


Всего пропусков delivery_days: 158
Пропуски по городам:
 city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Пропуски по статусам:
 status
оплачен    144
отменён     14
Актобе: 4 из 387, 1.03%
Алматы: 63 из 2116, 2.98%
Астана: 40 из 1339, 2.99%
Атырау: 5 из 143, 3.50%
Караганда: 24 из 519, 4.62%
Шымкент: 22 из 776, 2.84%
оплачен: 144 из 4739, 3.04%
отменён: 14 из 541, 2.59%
Шымкент: всего 776, известных сроков 754, пропусков 22, медиана 5 дней.
dropna: 225 / 754 = 29.84%
fillna(медиана): 225 / 776 = 28.99%
fillna(0): 247 / 776 = 31.83%
Для менеджера: честная доля среди заказов с известным сроком — 29,84% (225/754). По 22 заказам срок неизвестен; это не оценка безусловной доли всех заказов. Медиана 5 делает пропуски медленными (28,99%), а ноль — быстрыми (31,83%). Ни то ни другое не подтверждено данными; для SLA нужны фактические сроки завершённых доставок.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# Ошибка 1: результат drop_duplicates не был присвоен.
# Ошибка 2: lower не убирает крайние пробелы и не объединяет латиницу с кириллицей.
fixed = pd.read_csv(PATH)
fixed = fixed.drop_duplicates().copy()
fixed["city"] = fixed["city"].str.strip().str.lower().map(CITY)
assert fixed["city"].notna().all()
assert fixed["order_id"].is_unique
print("Исправленный фрагмент:", fixed["city"].nunique(), "городов;", len(fixed), "строк")
print("Для менеджера: 6 городов и 5 283 строки. Здесь исправлены ровно два дефекта "
      "фрагмента; после отдельного удаления 3 строк TEST будет 5 280 реальных заказов.")


Исправленный фрагмент: 6 городов; 5283 строк
Для менеджера: 6 городов и 5 283 строки. Здесь исправлены ровно два дефекта фрагмента; после отдельного удаления 3 строк TEST будет 5 280 реальных заказов.


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
ru = ["Понедельник", "Вторник", "Среда", "Четверг", "Пятница", "Суббота", "Воскресенье"]
counts = df["order_date"].dt.day_name().value_counts()
# date_range создаёт календарь; normalize убирает время, оставляя дату.
# В выборке 61 календарный день: сравним ещё и среднее число заказов за день недели.
calendar = pd.Series(pd.date_range(df["order_date"].min().normalize(),
                                 df["order_date"].max().normalize(), freq="D"))
day_counts = calendar.dt.day_name().value_counts()
for eng, rus in zip(weekday_order, ru):
    print(f"{rus}: {counts[eng]} заказов / {day_counts[eng]} дней = "
          f"{counts[eng] / day_counts[eng]:.2f} заказа в день")
# idxmin/idxmax возвращают название строки с минимальным/максимальным значением.
least, most = counts.idxmin(), counts.idxmax()
gap = (1 - counts[least] / counts[most]) * 100
print(f"Минимум: {least}, {counts[least]}; максимум: {most}, {counts[most]}; разница {gap:.2f}%.")
print("Для менеджера: в понедельник 684 заказа, в четверг 786 — на 12,98% меньше. "
      "Но понедельников 8, четвергов 9: на один день 85,50 против 87,33, разница лишь 2,10%. "
      "Резко сокращать смены в понедельник по суммам нельзя; планируйте по нагрузке за день, "
      "а график курьеров уточняйте по датам доставок и числу завершённых заказов.")


Понедельник: 684 заказов / 8 дней = 85.50 заказа в день
Вторник: 785 заказов / 9 дней = 87.22 заказа в день
Среда: 774 заказов / 9 дней = 86.00 заказа в день
Четверг: 786 заказов / 9 дней = 87.33 заказа в день
Пятница: 781 заказов / 9 дней = 86.78 заказа в день
Суббота: 773 заказов / 9 дней = 85.89 заказа в день
Воскресенье: 697 заказов / 8 дней = 87.12 заказа в день
Минимум: Monday, 684; максимум: Thursday, 786; разница 12.98%.
Для менеджера: в понедельник 684 заказа, в четверг 786 — на 12,98% меньше. Но понедельников 8, четвергов 9: на один день 85,50 против 87,33, разница лишь 2,10%. Резко сокращать смены в понедельник по суммам нельзя; планируйте по нагрузке за день, а график курьеров уточняйте по датам доставок и числу завершённых заказов.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
sh = df.loc[df["city"] == "Шымкент"].sort_values("order_date")
for month in [9, 10]:
    part = sh.loc[sh["month"] == month]
    print(f"Шымкент, месяц {month}: средний срок {part['delivery_days'].mean():.2f} дня; "
          f"известно {part['delivery_days'].notna().sum()} из {len(part)} сроков; "
          f"отмены {(part['status'] == 'отменён').mean():.2%}")
# Проверяем каждую дату по порядку: все последующие известные сроки должны быть >= 5.
known = sh.dropna(subset=["delivery_days"])
change_date = None
for date in known["order_date"].dt.normalize().drop_duplicates():
    after = known.loc[known["order_date"] >= date, "delivery_days"]
    if (after >= 5).all():
        change_date = date
        break
assert change_date is not None
last_fast = known.loc[known["delivery_days"] < 5, "order_date"].max()
first_slow = known.loc[known["order_date"] >= change_date, "order_date"].min()
print("Первая дата устойчивого перехода:", change_date.date())
print("Последний заказ со сроком <5:", last_fast)
print("Первый заказ после перехода:", first_slow)
after = sh.loc[sh["order_date"] >= change_date, "delivery_days"]
print(f"После перехода: сроки {after.min():.0f}–{after.max():.0f} дней, "
      f"неизвестных {after.isna().sum()}, известных меньше 5: {(after < 5).sum()}")
# Отдельно проверяем каждый другой город по месяцам и в соседние календарные даты.
for city in sorted(df.loc[df["city"] != "Шымкент", "city"].unique()):
    part = df.loc[df["city"] == city]
    means = [part.loc[part["month"] == month, "delivery_days"].mean() for month in [9, 10]]
    print(f"{city}: сентябрь {means[0]:.2f}; октябрь {means[1]:.2f} дня")
    for date in [change_date - pd.Timedelta(days=1), change_date]:
        d = part.loc[part["order_date"].dt.normalize() == date, "delivery_days"]
        print(f"  {date.date()}: n={d.notna().sum()}, среднее {d.mean():.2f}, "
              f"диапазон {d.min():.0f}–{d.max():.0f}")
print("Для менеджера: с 01.10.2026 известные сроки Шымкента выросли с 1–3 до 5–8 дней; "
      "среднее 2,03 → 6,54. У остальных городов в соседние даты диапазон 1–3 дня "
      "и месячные средние около 2 дней: аналогичного скачка не видно. "
      "Сначала проверьте смену перевозчика, маршрута, склада или правил учёта с 1 октября. "
      "Данные показывают момент изменения по дате заказа, но не доказывают его причину; "
      "проверка сроков относится к известным значениям, включая строки отмен.")


Шымкент, месяц 9: средний срок 2.03 дня; известно 349 из 359 сроков; отмены 8.91%
Шымкент, месяц 10: средний срок 6.54 дня; известно 405 из 417 сроков; отмены 30.70%
Первая дата устойчивого перехода: 2026-10-01
Последний заказ со сроком <5: 2026-09-30 22:00:00
Первый заказ после перехода: 2026-10-01 10:00:00
После перехода: сроки 5–8 дней, неизвестных 12, известных меньше 5: 0
Актобе: сентябрь 2.04; октябрь 2.10 дня
  2026-09-30: n=4, среднее 1.25, диапазон 1–2
  2026-10-01: n=4, среднее 2.00, диапазон 1–3
Алматы: сентябрь 1.98; октябрь 2.01 дня
  2026-09-30: n=19, среднее 2.00, диапазон 1–3
  2026-10-01: n=27, среднее 1.74, диапазон 1–3
Астана: сентябрь 2.05; октябрь 1.99 дня
  2026-09-30: n=25, среднее 1.72, диапазон 1–3
  2026-10-01: n=22, среднее 1.91, диапазон 1–3
Атырау: сентябрь 2.06; октябрь 2.01 дня
  2026-09-30: n=3, среднее 1.67, диапазон 1–2
  2026-10-01: n=1, среднее 2.00, диапазон 2–2
Караганда: сентябрь 1.90; октябрь 1.99 дня
  2026-09-30: n=9, среднее 2.11, диапазон 1–3

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
def to_amount(s):
    # StringDtype сохраняет пропуски как пропуски и поддерживает числовые входы.
    # Помимо обычного пробела встречаются неразрывный и узкий неразрывный.
    text = (s.astype("string").str.strip()
            .str.replace(" ", "", regex=False)
            .str.replace("\u00a0", "", regex=False)
            .str.replace("\u202f", "", regex=False)
            .str.replace(",", ".", regex=False))
    return pd.to_numeric(text, errors="coerce").astype("float64")

example = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
converted = to_amount(example)
print(pd.DataFrame({"Вход": example, "Результат": converted}).to_string(index=False))
assert converted.iloc[:3].tolist() == [15990.0, 15990.0, 1200.5]
assert pd.isna(converted.iloc[3])
amounts = to_amount(raw["amount"])
reference = pd.to_numeric(raw["amount"].str.replace(" ", "", regex=False))
assert amounts.notna().all()
assert amounts.sum() == reference.sum() == 109883360.0
print(f"Исходный CSV: сумма {amounts.sum():,.0f} ₸, пропусков {amounts.isna().sum()}.")
print("Для менеджера: сумма совпала с уроком — 109 883 360 ₸. Это все исходные строки "
      "со всеми статусами, включая TEST и дубли, а не чистая оплаченная выручка. "
      "Нераспознанные суммы становятся пропусками; перед отчётом их нужно проверять.")


      Вход  Результат
    15 990    15990.0
   15990.0    15990.0
  1 200,50     1200.5
нет данных        NaN
Исходный CSV: сумма 109,883,360 ₸, пропусков 0.
Для менеджера: сумма совпала с уроком — 109 883 360 ₸. Это все исходные строки со всеми статусами, включая TEST и дубли, а не чистая оплаченная выручка. Нераспознанные суммы становятся пропусками; перед отчётом их нужно проверять.
